# Day 26: Classification Metrics Deep Dive

**Dataset:** `Customer_Churn_Dataset.csv`

This notebook evaluates a classification model using accuracy, precision, recall, F1-score,
and support, and determines when each metric is useful, using real values from a customer
churn dataset (predicting whether a customer will `Churn`, 1 = yes, 0 = no).

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Load the dataset
df = pd.read_csv("../datasets/Customer_Churn_Dataset.csv")

# Preview the data
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [2]:
print(df.shape)
print(df.isna().sum().sum(), "missing values")
print()
print("Target balance:")
print(df["Churn"].value_counts(normalize=True).round(3))

(64374, 12)
0 missing values

Target balance:
Churn
0    0.526
1    0.474
Name: proportion, dtype: float64


## 2. Preprocessing and Training a Model to Evaluate

In [3]:
df = df.drop(columns=["CustomerID"])
df = pd.get_dummies(df, columns=["Gender", "Subscription Type", "Contract Length"], drop_first=True)

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_columns = ["Age", "Tenure", "Usage Frequency", "Support Calls", "Payment Delay", "Total Spend", "Last Interaction"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
X_test_scaled[numeric_columns] = scaler.transform(X_test[numeric_columns])

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("Model trained and predictions generated.")

Model trained and predictions generated.


## 3. Understanding False Positives and False Negatives

In [4]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print(f"True Negatives (TN):  {tn}  - correctly predicted NOT churn")
print(f"False Positives (FP): {fp}  - predicted churn, but customer actually stayed")
print(f"False Negatives (FN): {fn}  - predicted NOT churn, but customer actually churned")
print(f"True Positives (TP):  {tp}  - correctly predicted churn")

True Negatives (TN):  5628  - correctly predicted NOT churn
False Positives (FP): 1148  - predicted churn, but customer actually stayed
False Negatives (FN): 1077  - predicted NOT churn, but customer actually churned
True Positives (TP):  5022  - correctly predicted churn


## 4. Classification Report

In [5]:
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

              precision    recall  f1-score   support

    No Churn       0.84      0.83      0.83      6776
       Churn       0.81      0.82      0.82      6099

    accuracy                           0.83     12875
   macro avg       0.83      0.83      0.83     12875
weighted avg       0.83      0.83      0.83     12875



## 5. Metric Comparison

Calculating each metric manually from the confusion matrix counts, to see exactly what each
one measures, then confirming they match `scikit-learn`'s built-in functions.

In [6]:
manual_accuracy = (tp + tn) / (tp + tn + fp + fn)
manual_precision = tp / (tp + fp)
manual_recall = tp / (tp + fn)
manual_f1 = 2 * (manual_precision * manual_recall) / (manual_precision + manual_recall)

print(f"Accuracy:  manual={manual_accuracy:.3f}  sklearn={accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: manual={manual_precision:.3f}  sklearn={precision_score(y_test, y_pred):.3f}")
print(f"Recall:    manual={manual_recall:.3f}  sklearn={recall_score(y_test, y_pred):.3f}")
print(f"F1-score:  manual={manual_f1:.3f}  sklearn={f1_score(y_test, y_pred):.3f}")

Accuracy:  manual=0.827  sklearn=0.827
Precision: manual=0.814  sklearn=0.814
Recall:    manual=0.823  sklearn=0.823
F1-score:  manual=0.819  sklearn=0.819


In [7]:
metric_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Formula": [
        "(TP + TN) / Total",
        "TP / (TP + FP)",
        "TP / (TP + FN)",
        "2 * (Precision * Recall) / (Precision + Recall)"
    ],
    "What it answers": [
        "Overall, what fraction of predictions were correct?",
        "Of everyone predicted to churn, how many actually churned?",
        "Of everyone who actually churned, how many did we catch?",
        "A single balance between precision and recall"
    ],
    "Value": [manual_accuracy, manual_precision, manual_recall, manual_f1]
})

metric_comparison

,Metric,Formula,What it answers,Value
0,Accuracy,(TP + TN) / Total,"Overall, what fraction of predictions were cor...",0.827184
1,Precision,TP / (TP + FP),"Of everyone predicted to churn, how many actua...",0.813938
2,Recall,TP / (TP + FN),"Of everyone who actually churned, how many did...",0.823414
3,F1-score,2 * (Precision * Recall) / (Precision + Recall),A single balance between precision and recall,0.818649


## 6. Why Accuracy Isn't Always the Best Metric

A quick demonstration: a "naive" model that always predicts the majority class, with no real
learning at all.

In [8]:
majority_class = y_train.mode()[0]
naive_predictions = np.full(len(y_test), majority_class)

print("Naive baseline accuracy:", accuracy_score(y_test, naive_predictions))
print()
print(classification_report(y_test, naive_predictions, target_names=["No Churn", "Churn"], zero_division=0))

Naive baseline accuracy: 0.5262912621359224



              precision    recall  f1-score   support

    No Churn       0.53      1.00      0.69      6776
       Churn       0.00      0.00      0.00      6099

    accuracy                           0.53     12875
   macro avg       0.26      0.50      0.34     12875
weighted avg       0.28      0.53      0.36     12875



**Observation:** even on this fairly balanced dataset (≈53% No Churn / 47% Churn), a model
that always predicts "No Churn" still gets about **52.6% accuracy** — while having **0%
recall** for the Churn class, meaning it catches exactly zero actual churners. Accuracy alone
makes this naive, useless model look only moderately bad; precision and recall immediately
expose that it has no real predictive value for the class that actually matters here. On a more
imbalanced dataset, this effect would be even more dramatic.

## 7. Business / Practical Interpretation of These Metrics

In a customer churn scenario, False Positives and False Negatives have very different
real-world costs:

- **A False Positive** (predicting a customer will churn, when they actually wouldn't have)
  means the business spends retention resources (e.g., a discount, a call, a special offer) on a
  customer who was never going to leave. This costs money and effort, but it's a relatively
  **low-cost mistake**.
- **A False Negative** (predicting a customer will stay, when they actually churn) means the
  business does nothing, and then **loses that customer entirely**, along with all their
  future revenue. This is typically a **much more expensive mistake** than a False Positive,
  since acquiring a new customer to replace a lost one is usually far costlier than the price
  of an unnecessary retention offer.

Because False Negatives tend to be the costlier error here, **recall** (catching as many real
churners as possible) usually deserves more weight than precision in this kind of business
problem, even if that means tolerating somewhat more False Positives along the way.

## 8. Recommendation of the Most Suitable Metric

**Recommended metric: Recall for the "Churn" class (supported by F1-score as a secondary check).**

- **Recall** should be the primary metric to optimize, since missing an actual churner (a
  False Negative) is the costlier mistake in this business context, thus, every uncaught churner is
  lost revenue the business never gets a chance to try to save.
- **F1-score** is still worth tracking alongside recall, as a safeguard against chasing recall
  so aggressively that precision collapses (i.e., flagging almost everyone as a likely
  churner, which would waste retention resources on customers who were never at risk).
- **Accuracy alone should not be the deciding metric** since, as shown above, a model with no real
  skill can still post a deceptively reasonable-looking accuracy number, while completely
  failing at the one thing that actually matters for this problem: catching real churners
  before they leave.

## Outcome

This notebook evaluated a logistic regression churn-prediction model using accuracy,
precision, recall, F1-score, and the confusion matrix counts behind them, calculating each
metric manually from True/False Positives/Negatives and confirming the results matched
`scikit-learn`'s built-in functions. A naive always-predict-majority-class baseline
demonstrated concretely why accuracy alone can be misleading, i.e., it scored a deceptively
reasonable 52.6% while having zero ability to catch actual churners. Considering the real
business costs of each error type — a False Positive wastes a retention offer, while a False
Negative loses the customer and their future revenue entirely — led to recommending recall as
the primary metric for this problem, with F1-score as a secondary check against chasing recall
at the expense of precision. This showed why choosing the right evaluation metric depends on
the real-world cost of each kind of mistake, not just on which metric looks highest.